In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

# Load data
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/monthly-airline-passengers.csv"
data = pd.read_csv(url).iloc[:,1].values.astype(float)

scaler = MinMaxScaler()
data = scaler.fit_transform(data.reshape(-1,1))

# Create sequences
X = np.array([data[i:i+10] for i in range(len(data)-10)])
y = np.array([data[i+10] for i in range(len(data)-10)])

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

# BiRNN
class BiRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(1, 32, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1])

# FFNN
class FFNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(10, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.fc(x.view(x.size(0), -1))

# Train
def train(model):
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    loss = nn.MSELoss()
    for i in range(50):
        opt.zero_grad()
        l = loss(model(X), y)
        l.backward()
        opt.step()

rnn, ffnn = BiRNN(), FFNN()
train(rnn)
train(ffnn)

# MSE
with torch.no_grad():
    p1 = rnn(X).numpy()
    p2 = ffnn(X).numpy()

    print("BiRNN MSE:", mean_squared_error(y, p1))
    print("FFNN MSE:", mean_squared_error(y, p2))

BiRNN MSE: 0.0027995151467621326
FFNN MSE: 0.008005906827747822


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

# Data
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/monthly-airline-passengers.csv"
data = pd.read_csv(url, usecols=[1]).values.astype("float32")

scaler = MinMaxScaler()
data = scaler.fit_transform(data)

# Sequences
seq = 10
X = np.array([data[i:i+seq] for i in range(len(data)-seq)])
y = np.array([data[i+seq] for i in range(len(data)-seq)])

n = int(0.8 * len(X))
X_train, X_test = torch.tensor(X[:n]), torch.tensor(X[n:])
y_train, y_test = torch.tensor(y[:n]), torch.tensor(y[n:])

# BiRNN
class BiRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(1, 32, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

# FFNN
class FFNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(10, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.fc(x.view(x.size(0), -1))

# Training
def train(model):
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.MSELoss()

    for _ in range(50):
        opt.zero_grad()
        loss = loss_fn(model(X_train), y_train)
        loss.backward()
        opt.step()

birnn, ffnn = BiRNN(), FFNN()
train(birnn)
train(ffnn)

# Prediction and MSE
with torch.no_grad():
    p1 = birnn(X_test).numpy()
    p2 = ffnn(X_test).numpy()

    print("BiRNN MSE:", mean_squared_error(y_test, p1))
    print("FFNN MSE:", mean_squared_error(y_test, p2))

BiRNN MSE: 0.006277583073824644
FFNN MSE: 0.014561446383595467
